# Questions 5 and 6 — Abdelrahman Elakkad

Two research questions for the shared story: completed affordable housing and serious road crashes in New York City, joined by council district, for 2022 to 2025. Both charts describe an association. Neither chart shows that housing causes crashes, or the reverse.

The cleaned files are the ones saved by the team notebook: `panel_district_year.csv`, `housing_clean.csv`, and `crashes_clean.csv`.


In [1]:
from pathlib import Path
import subprocess
import pandas as pd
import plotly.express as px

def find_data_dir():
    """Locate the cleaned Data folder locally, or clone the team repo on Colab."""
    cwd = Path.cwd()
    candidates = [
        cwd / "Data",
        cwd.parent / "Data",
        cwd / "data-engineering-project-1" / "Data",
        Path("/content/data-engineering-project-1/Data"),
    ]
    for path in candidates:
        if (path / "panel_district_year.csv").exists():
            return path
    clone_to = Path("/content/data-engineering-project-1")
    if not clone_to.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/ahmed-roushdy16004392/data-engineering-project-1.git",
            str(clone_to),
        ])
    return clone_to / "Data"

DATA = find_data_dir()
panel = pd.read_csv(DATA / "panel_district_year.csv")
housing = pd.read_csv(DATA / "housing_clean.csv")
crashes = pd.read_csv(DATA / "crashes_clean.csv")
print("Data folder:", DATA)
print("panel", panel.shape, "| housing", housing.shape, "| crashes", crashes.shape)


Data folder: /Users/abdelrahmanakkad/Desktop/data-eng-proj/data-engineering-project-1/Data
panel (204, 12) | housing (2957, 12) | crashes (11958, 15)


### Question 5: Do districts with more affordable units also tend to have more severe-injury crashes?

**Member:** Abdelrahman Elakkad
**Data used:** `panel_district_year.csv`, summed to one row per council district
**Chart:** scatter plot, one dot per district, coloured by borough


In [2]:
# Question 5
# One row per council district: the panel is district-by-year, and the question compares districts.
districts = (
    panel.groupby(["council_final", "borough"], as_index=False)[["affordable_units", "severe_injuries"]]
    .sum()
)
districts["council_final"] = districts["council_final"].astype(int)

rho = districts["affordable_units"].rank().corr(districts["severe_injuries"].rank())
most = districts.nlargest(13, "affordable_units")["severe_injuries"].mean()
least = districts.nsmallest(13, "affordable_units")["severe_injuries"].mean()

# Subtract each district's own average, so this correlation is about years inside a
# district rather than about which districts are larger.
demeaned = panel.copy()
for col in ["affordable_units", "severe_injuries"]:
    demeaned[col] = demeaned[col] - demeaned.groupby("council_final")[col].transform("mean")
within_rho = demeaned["affordable_units"].rank().corr(demeaned["severe_injuries"].rank())

print(f"Districts: {len(districts)}")
print(f"Spearman rank correlation, across districts: {rho:.2f}")
print(f"Mean severe injuries, 13 districts with the most units: {most:.0f}")
print(f"Mean severe injuries, 13 districts with the least units: {least:.0f}")
print(f"Spearman rank correlation, year-to-year inside a district: {within_rho:.2f}")

# Label the districts discussed in the write-up. The rest stay unlabeled so the chart is readable.
label_ids = {12, 8, 3, 4, 31}
districts["label"] = districts["council_final"].map(lambda d: str(d) if d in label_ids else "")
print(districts.loc[districts["council_final"].isin(label_ids),
                    ["council_final", "borough", "affordable_units", "severe_injuries"]]
      .sort_values("affordable_units", ascending=False)
      .to_string(index=False))

fig = px.scatter(
    districts,
    x="affordable_units",
    y="severe_injuries",
    color="borough",
    text="label",
    hover_name="council_final",
    title=f"Council districts, 2022-2025: affordable units and severe injuries (rank correlation {rho:.2f})",
    labels={
        "affordable_units": "Affordable units completed",
        "severe_injuries": "Severe injuries",
        "borough": "Borough",
        "council_final": "Council district",
    },
    category_orders={"borough": ["Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"]},
    color_discrete_map={
        "Bronx": "#E45756",
        "Brooklyn": "#4C78A8",
        "Manhattan": "#F58518",
        "Queens": "#54A24B",
        "Staten Island": "#B279A2",
    },
)
fig.update_traces(textposition="top center", marker={"size": 11})
fig.add_vline(x=districts["affordable_units"].median(), line_dash="dash", line_color="grey")
fig.add_hline(y=districts["severe_injuries"].median(), line_dash="dash", line_color="grey")
fig.update_layout(template="plotly_white", height=560, width=900, title_x=0.5)
fig.show()


Districts: 51
Spearman rank correlation, across districts: 0.50
Mean severe injuries, 13 districts with the most units: 245
Mean severe injuries, 13 districts with the least units: 170
Spearman rank correlation, year-to-year inside a district: 0.05
 council_final   borough  affordable_units  severe_injuries
            12     Bronx             17064              208
             8     Bronx              5662              358
            31    Queens              3921              177
             3 Manhattan              1885              324
             4 Manhattan               733              259


**Why this chart:** both measures are district totals, so one dot per district shows whether high housing and high injuries tend to land on the same districts. Borough colour shows whether that pattern belongs to one borough. The dashed lines are the medians. Numbered dots are the districts named below.

**What the chart shows:** They tend to, loosely. Across the 51 districts the Spearman rank correlation is 0.50. The 13 districts with the most affordable units average 245 severe injuries, against 170 for the 13 with the least. The Bronx occupies much of the upper right. District 8 is high on both: 5,662 affordable units and 358 severe injuries, the most injuries of any district. District 12 is the clear exception: 17,064 units, far more than any other district, but 208 severe injuries, close to the middle of the city. District 3 in Manhattan is the other injury peak (324) with many fewer units (1,885). District 4 (Manhattan) is also high on injuries (259) with only 733 units, while district 31 (Queens) has 3,921 units and only 177 severe injuries. The same pairing does not show up over time. After each district's own average is removed, year-to-year changes in units and injuries have a rank correlation of 0.05, so a year with more housing inside a district was not a year with more injuries.

**Limits:** These are counts, not rates. A larger or busier district can be high on both without affordable housing causing crashes. Crashes that could not be placed in a district are missing from these totals, and fatal crashes are left out because most of them have no district.


### Question 6: Where do severe-injury crashes and new-construction buildings sit on the same map?

**Member:** Abdelrahman Elakkad
**Data used:** `crashes_clean.csv` and `housing_clean.csv`
**Chart:** dot map, red for severe-injury crashes and blue for new-construction affordable buildings


In [3]:
# Question 6
# Keep severe-injury crashes only. Fatal crashes are a separate severity and are mostly missing coordinates.
severe = crashes[crashes["severity"].eq("Severe injury")].copy()
new_buildings = housing[housing["Reporting Construction Type"].eq("New Construction")].copy()

severe_mapped = severe.dropna(subset=["Latitude", "Longitude"])
buildings_mapped = new_buildings.dropna(subset=["Latitude", "Longitude"])

print(f"Severe-injury crashes: {len(severe):,} | with coordinates: {len(severe_mapped):,} "
      f"({100 * severe['Latitude'].isna().mean():.0f}% missing)")
print(f"New-construction buildings: {len(new_buildings):,} | with coordinates: {len(buildings_mapped):,} "
      f"({100 * new_buildings['Latitude'].isna().mean():.0f}% missing)")
print("Mapped new-construction buildings by borough:")
print(buildings_mapped["borough_clean"].value_counts().to_string())
print("Mapped severe-injury crashes by borough:")
print(severe_mapped["borough_clean"].value_counts().to_string())

top_buildings = set(buildings_mapped.groupby("council_final").size().nlargest(10).index.astype(int))
top_crashes = set(severe_mapped.dropna(subset=["council_final"]).groupby("council_final").size().nlargest(10).index.astype(int))
print("Top 10 districts by new-construction buildings:", sorted(top_buildings))
print("Top 10 districts by severe-injury crashes:", sorted(top_crashes))
print("In both top 10s:", sorted(top_buildings & top_crashes))

points = pd.concat([
    severe_mapped.assign(place="Severe-injury crash"),
    buildings_mapped.assign(place="New-construction building"),
], ignore_index=True)

fig = px.scatter_map(
    points,
    lat="Latitude",
    lon="Longitude",
    color="place",
    hover_data={"borough_clean": True, "council_final": True, "year": True, "Latitude": False, "Longitude": False},
    zoom=9,
    center={"lat": 40.71, "lon": -73.98},
    map_style="open-street-map",
    title="Severe-injury crashes and new-construction affordable buildings, 2022-2025",
    labels={"place": "", "borough_clean": "Borough", "council_final": "Council district", "year": "Year"},
    color_discrete_map={
        "Severe-injury crash": "#E45756",
        "New-construction building": "#4C78A8",
    },
    opacity=0.65,
    height=700,
)
fig.update_traces(marker={"size": 4}, selector={"name": "Severe-injury crash"})
fig.update_traces(marker={"size": 8}, selector={"name": "New-construction building"})
fig.update_layout(margin={"l": 0, "r": 0, "t": 50, "b": 0}, title_x=0.5)
fig.show()


Severe-injury crashes: 11,033 | with coordinates: 9,477 (14% missing)
New-construction buildings: 1,886 | with coordinates: 1,460 (23% missing)
Mapped new-construction buildings by borough:
borough_clean
Brooklyn         662
Bronx            492
Queens           216
Manhattan         85
Staten Island      5
Mapped severe-injury crashes by borough:
borough_clean
Brooklyn         2890
Queens           2359
Manhattan        1907
Bronx            1878
Staten Island     442
Unknown             1
Top 10 districts by new-construction buildings: [14, 15, 16, 17, 34, 36, 37, 40, 41, 45]
Top 10 districts by severe-injury crashes: [3, 4, 8, 15, 16, 17, 20, 26, 33, 41]
In both top 10s: [15, 16, 17, 41]


**Why this chart:** the question is about place. A map shows whether the two sets of points occupy the same parts of the city, which a district total cannot show. Blue dots are drawn larger so the buildings stay visible on top of the crashes.

**What the chart shows:** Severe-injury crashes that have coordinates (9,477 of 11,033) cover every borough. Of those mapped crashes, 30% are in Brooklyn, 25% in Queens, 20% in Manhattan, and 20% in the Bronx. New-construction affordable buildings that have coordinates (1,460 of 1,886) are more concentrated: 45% of the mapped buildings are in Brooklyn (662) and 34% are in the Bronx (492). Queens has the second-most mapped crashes but only 15% of the mapped new-construction buildings. The two sets overlap in the Bronx and in parts of Brooklyn. Districts 15, 16, 17, and 41 are in the top 10 both for mapped new-construction buildings and for mapped severe-injury crashes. Manhattan districts 3 and 4 are among the highest for crashes and are not among the highest for these buildings. Sharing a map shows a shared location only.

**Limits:** 14% of severe-injury crashes and 23% of new-construction buildings have no coordinates, so they are missing. That gap is uneven: Staten Island has 78 new-construction buildings in the cleaned file and only 5 of them have coordinates, so the map almost leaves that borough's new buildings out. Preservation buildings are not on this map. Dots are not adjusted for street length or traffic, and a cluster is not evidence that new buildings cause crashes.
